# Introduction Approach 1

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **multiple years (pooled together)**, **NOT** separate GEVs for each model <br>
-> Separate analysis only for different locations <br>


**Additional Notes**
- Using sim_year as the actual year 

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

**!!! ToDo**
- plot model fits
- create a mapping function for location info and lat/lon... store as JSON to save lookup
- analyze using all simulation at one location from all models (Approach2 - see email 26.01.26)
- add Confidence Intervals for Return Levels
- add return level (and CI) to txt
- **check extract_annual_max - is function correct?**

# Import Libraries

In [1]:
import multiprocessing as mp
import random
import time
from datetime import datetime
from glob import glob
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import psutil
import xarray as xr
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from numpy import isnan, unique
from pandas import DataFrame, concat

import func_gev as gev
import func_plotting as dbplt
import func_preparation as dbf
import func_utils as ut

# Settings

In [2]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [3]:
hindcast_start = 1960
hindcast_end = 2026

In [4]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [5]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

For code parallelization later in the process 

In [6]:
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"Logical cores (mp): {mp.cpu_count()}")

Physical cores: 4
Logical cores: 8
Logical cores (mp): 8


# Import data

In [7]:
ls_files = [file for file in glob(path + '*.nc')]
ls_files

['../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc']

In [8]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

Importing data from model MIROC6 (1/8)...
Importing data from model MPI-ESM1-2-HR (2/8)...
Importing data from model HadGEM3-GC31-MM (3/8)...
Importing data from model MRI-ESM2-0 (4/8)...
Importing data from model BCC-CSM2-MR (5/8)...
Importing data from model CMCC-CM2-SR5 (6/8)...
Importing data from model CanESM5 (7/8)...
Importing data from model NorCPM1 (8/8)...


# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, trying first with `joblib` - saving ~60% (from 3min30sec down to 1min22sec)


In [9]:
def process_model(file):
    model_name, ds_model = dbf.import_data_from_file(file) 
    ds_model_corrected = dbf.bias_correction(ds_model)
    data_valid, sites_valid, sites_total, rate_invalid = dbf.select_valid_data(
        ds_model, ds_model_corrected
    )
    ds_model.close()
    return model_name, data_valid, (sites_valid, sites_total, rate_invalid)

# ----------------------------------------------------------------------------- 
time_start1 = datetime.now()

results = Parallel(n_jobs=4)(
    delayed(process_model)(file) for file in ls_files
)

for model_name, data_valid, prep_info in results:
    dic_data_per_model[model_name]['valid data'] = data_valid
    dic_data_per_model[model_name]['preparation info'] = prep_info
    
time_end1 = datetime.now()

In [12]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))

ls_num_samples = []
dic_sim_years_per_model = dict()
for model_label in dic_data_per_model.keys():
    years = dic_data_per_model[model_label]['valid data'].sim_year.values
    unique_years = unique(years[~isnan(years)].astype(int))
    dic_sim_years_per_model[model_label] = unique_years
    data_shape = dic_data_per_model[model_label]['valid data'].shape
    
    ls_num_samples.append(data_shape[0])
    print(f"{model_label} · {data_shape}")
    
sites_valid = [dic_data_per_model[model_label]['preparation info'][0] for model_label in dic_data_per_model.keys()]
unique_years_per_model = [len(dic_sim_years_per_model[model_label]) for model_label in dic_sim_years_per_model.keys()]
sim_year_per_model_min = [min(dic_sim_years_per_model[model_label]) for model_label in dic_sim_years_per_model.keys()]
sim_year_per_model_max = [max(dic_sim_years_per_model[model_label]) for model_label in dic_sim_years_per_model.keys()]

print(
    "\nOverall, data is available from "
    f"\n\t{len(ls_files)} models, "
    f"\n\t{min(sites_valid)}-{max(sites_valid)} locations "
    f"(originally {dic_data_per_model[model_label]['preparation info'][1]})"
    f"\n\t{min(ls_num_samples)}-{max(ls_num_samples)} samples per model "
    f"\n\t - with {min(unique_years_per_model)}-{max(unique_years_per_model)} unique sim_years"
    f"\n\t - between {min(sim_year_per_model_min)}-{max(sim_year_per_model_max)}"
    )
display(Markdown(f"Execution time · {time_end1 - time_start1}sec"))

**Data Overview**

**Model · model shape: samples (~sim_years) | ensemble members | valid locations**

MIROC6 · (680, 2, 7054)
MPI-ESM1-2-HR · (630, 2, 5808)
HadGEM3-GC31-MM · (680, 2, 7216)
MRI-ESM2-0 · (320, 2, 3547)
BCC-CSM2-MR · (630, 2, 5236)
CMCC-CM2-SR5 · (690, 2, 6038)
CanESM5 · (660, 2, 5782)
NorCPM1 · (630, 2, 4558)

Overall, data is available from 
	8 models, 
	3547-7216 locations (originally 11022)
	320-690 samples per model 
	 - with 63-69 unique sim_years
	 - between 1961-2029


Execution time · 0:01:10.816252sec

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)

Also here, parallelize the code to speed up the process...

In [ ]:
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
print("Final dimensions:", combined.dims)
print("Shape:", combined.shape)
print("Number of models:", combined.model.size)
print("Number of locations:", combined.location.size)

/var/folders/lx/z70mzvpx4ls9np3hfbhll4wr0000gn/T/ipykernel_7608/2393049472.py:10: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  combined = xr.concat(da_list, dim="model", join="outer")



**Overall, the combined dataset has the following dimensions**

Final dimensions: ('model', 'sample', 'member', 'location')
Shape: (8, 690, 2, 9589)
Number of models: 8
Number of locations: 9589


### Validation Check

In [14]:
list_model_labels = list(dic_data_per_model.keys())

In [15]:
model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

For validity check, use randomly selected model BCC-CSM2-MR and site-id 1582...



**Overview Original dataArray**

Model BCC-CSM2-MR (None) - location-ID 1582 
Full dataframe (630, 2) vs reduced (537, 2)
coordinates in original dataset lon|lat: 33.74954|36.20054



**Overview Revised dataArray**

Model None (4) - location-ID None 
Full dataframe (690, 2) vs reduced (537, 2)
coordinates in original dataset lon|lat: 33.74954|36.20054


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:258: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [16]:
assert lon_target == lon_rev and lat_target == lat_rev
assert all(data_for_model_for_location.dropna() == revised_dataset.dropna())

**Learning** · make sure to never select by the site-id but always via geo-coordinates (lat | lon)

# Workflow GEV - Generalized Extreme Value

## Initial trial at 1 location, all model simulations and years

Later, batch(?) and parallelize

In [17]:
columns_selected = ['sim_year', 'annualMax', 'lon', 'lat', 'member', 'model']

In [18]:
loc_ex = 0

In [19]:
data_at_location = combined[:,:,:, loc_ex].to_dataframe().dropna().reset_index()[columns_selected]
data_at_location = data_at_location.rename(columns={'annualMax':'storm_surge'})
data_at_location

,sim_year,storm_surge,lon,lat,member,model
0,1961.0,0.155305,-18.157674,27.731363,1,MIROC6
1,1961.0,0.216125,-18.157674,27.731363,2,MIROC6
2,1962.0,0.184408,-18.157674,27.731363,1,MIROC6
3,1962.0,0.138211,-18.157674,27.731363,2,MIROC6
4,1962.0,0.184914,-18.157674,27.731363,1,MIROC6
...,...,...,...,...,...,...
8279,2027.0,0.184923,-18.157674,27.731363,2,NorCPM1
8280,2027.0,0.088951,-18.157674,27.731363,1,NorCPM1
8281,2027.0,0.103984,-18.157674,27.731363,2,NorCPM1
8282,2028.0,0.123450,-18.157674,27.731363,1,NorCPM1


---
To be continued

## Run Analysis

In [21]:
time_start = time.time()

print("\n" + "="*70)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER MODEL")
print("="*70)

# ---------------------------------------------
df_prepared = ut.prepare_pooled_data(data=data_at_location, hindcast_start=hindcast_start, hindcast_end=hindcast_end)

lon = data_at_location.lon.unique()[0]
lat = data_at_location.lat.unique()[0]

# ---------------------------------------------
results = {}
print(f"\nAnalyzing specific location with data from {df_prepared.model.nunique()} model(s) ...")

print("\tLookup location info for batch...")
locations_label = dbf.locations_label_lookup_batched(DataFrame([lon, lat], index=['lon', 'lat']).T)
location_info = locations_label[0]
print(f"\n\tAnalyse location {location_info[0]}")

result = gev.analyze_per_location(df_prepared, lat, lon, location_info[0], return_periods)

if result is None:
    print(f"\t\t→ Warning! No results found, skipping...")
else:
    results[(lat, lon)] = result
    print(f"\t\t→ Results produced successfully; storing to dictionary...")

print("\n" + "="*70)
time_end = time.time()
print(f"✓ ANALYSIS COMPLETED IN {(time_end - time_start):.2f}s!")
print("="*70)


STORM SURGE GEV ANALYSIS - POOLED APPROACH PER MODEL

Data Summary:
  Hindcast period: 1960-2026
  Total observations: 8,248
  Models: 8
  Locations: 1

Analyzing specific location with data from 8 model(s) ...
	Lookup location info for batch...
	✓ Loaded 111 cached results
	Processing batch 1/1 (locations 0-1)


	Batch 1: 100%|██████████| 1/1 [00:00<00:00, 1543.16it/s]

	✓ Batch 1 saved to cache
	✓ All batches complete: 1/1 successful

	Analyse location Loma del Agua Azul, Frontera, Santa Cruz de Tenerife, Canarias, España
		conducting stationary GEV...


			stationary GEV done (success True); continuing with non-stationary GEV...
			non-stationary GEV done (success True).
		→ Results produced successfully; storing to dictionary...

✓ ANALYSIS COMPLETED IN 0.78s!


# Display results for one location


In [22]:
example_location = list(results.keys())[0]
example_location

(np.float64(27.731362916530976), np.float64(-18.15767355838487))

In [23]:
result_display = results[example_location]
location_in_example = result_display['location info']

display(Markdown("\n**Subset Description**"))
print(f"number of items: {len(result_display)}")
print(f"keys: {result_display.keys()}")
print(f"values: {result_display.values()}")


**Subset Description**

number of items: 10
keys: dict_keys(['location', 'location info', 'annual_maxima', 'gev_stationary', 'gev_nonstationary', 'model_comparison', 'return_levels_stationary', 'return_levels_nonstationary_start', 'return_levels_nonstationary_end', 'data from model(s)'])
values: dict_values([(np.float64(27.731362916530976), np.float64(-18.15767355838487)), 'Loma del Agua Azul, Frontera, Santa Cruz de Tenerife, Canarias, España',         year  annual_max        lon        lat  member         model
0     1961.0    0.110599 -18.157674  27.731363       1   BCC-CSM2-MR
1     1961.0    0.146076 -18.157674  27.731363       2   BCC-CSM2-MR
2     1961.0    0.173618 -18.157674  27.731363       1  CMCC-CM2-SR5
3     1961.0    0.127081 -18.157674  27.731363       2  CMCC-CM2-SR5
4     1961.0    0.184663 -18.157674  27.731363       1       CanESM5
...      ...         ...        ...        ...     ...           ...
8243  2026.0    0.164581 -18.157674  27.731363       2       NorCPM1
8244  2026.0    0.1315

#### Display Result Overview and Plots (all)

In [26]:
ls_messages = []
print_msg = True

if result_display:
    ls_messages = ut.adding_plot_and_text("\n" + "="*100, ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"RESULTS for {location_in_example}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text("="*100, ls_messages, print_msg)
    
    ls_messages = ut.adding_plot_and_text(f"\nlocation (lon|lat): {result_display['location'][1]:.5f}|{result_display['location'][0]:.5f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(
        f"Available simulation years: {result_display['annual_maxima'].year.min().astype(int)}-{result_display['annual_maxima'].year.max().astype(int)}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"Observations per year: ~2 (from ensemble members)", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"Total data points for GEV: {result_display['gev_stationary']['n_obs']}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text("\nSTATIONARY GEV", ls_messages, print_msg)
    stat = result_display['gev_stationary']
    ls_messages = ut.adding_plot_and_text(f"  μ (location) = {stat['location']:.3f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  σ (scale) = {stat['scale']:.3f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  ξ (shape) = {stat['shape']:.3f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  Type: {stat['dist_type']}", ls_messages, print_msg)

    if result_display['gev_nonstationary']:
        ls_messages = ut.adding_plot_and_text("\nNON-STATIONARY GEV", ls_messages, print_msg)
        nonstat = result_display['gev_nonstationary']
        ls_messages = ut.adding_plot_and_text(f"  μ(t) = {nonstat['mu0']:.3f} + {nonstat['mu1']:.4f}·t", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  Trend = {nonstat['mu1'] * nonstat['years_std']:.4f} m/year", ls_messages, print_msg)

    if result_display['model_comparison']:
        ls_messages = ut.adding_plot_and_text("\nMODEL COMPARISON", ls_messages, print_msg)
        comp = result_display['model_comparison']
        ls_messages = ut.adding_plot_and_text(f"  p-value: {comp['p_value']:.4f}", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  Decision: {comp['decision']}", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  → {comp['recommendation']}", ls_messages, print_msg)

    # -----------------------------------------------------------------------------------------------------------
    today_ = str(datetime.today().date().isoformat())
    save_path = path_export + today_
    Path(save_path).mkdir(parents=True, exist_ok=True)            
    country = result_display['location info'].split(',')[-1].strip()
    lat_str = str(round(float(result_display['location'][0]), 3))
    lon_str = str(round(float(result_display['location'][1]),3))
    
    #with open(save_path + f"/GEVanalysis_{country}_{lat_str}|{lon_str}_{today_}.txt", 'w') as f:
    #    f.write('\n'.join(ls_messages))
    
    print("\nVISUALIZE RESULTS")
    


RESULTS for Loma del Agua Azul, Frontera, Santa Cruz de Tenerife, Canarias, España

location (lon|lat): -18.15767|27.73136
Available simulation years: 1961-2026
Observations per year: ~2 (from ensemble members)
Total data points for GEV: 8248

STATIONARY GEV
  μ (location) = 0.126
  σ (scale) = 0.032
  ξ (shape) = -0.110
  Type: Weibull (Type III)

NON-STATIONARY GEV
  μ(t) = 0.126 + -0.0002·t
  Trend = -0.0032 m/year

MODEL COMPARISON
  p-value: 0.5895
  Decision: No strong evidence for non-stationarity
  → Use stationary model (simpler)

VISUALIZE RESULTS


---

to be continued - update plot

In [ ]:
plot_analysis_per_location(
        results=results, 
        model=None, 
        lat_lon_tuple=result_display['location'], 
        location_info=result_display['location info'],
        periods_evolution = plot_period_evolution,
        box_parameters_x=0.05, box_parameters_y=0.95,
        width_bar_returns=0.35,
        leg_comparison_x=0.35, leg_comparison_y=0.65, linespace=1.5,
        save_path = save_path,
        color_markers='#99E3DDFF', 
        colors_trends='#1D141BFF', 
        colors_models=['#B887ADFF', '#008A80FF'],
        colors_return_levels=['#008A80FF','#CAA5C2FF'],
        bbox_color='#F5F5F5FF',
        axes_color='#333333', 
        linestyle_trends=['dashdot', 'dashed', 'solid'], 
        fontsize=12, figsize=(15, 7.5),
    )